# NeuroGuard: Phase 2 data audit and patient splits
This project is a prototype and is not medical advice.

Canonical source: this notebook and src/ in the local Git repository. Colab executes a fixed Git commit; opening this file from GitHub does not clone its modules. Do not edit code in Colab and expect local files to change.

Run cells 1–8 in order. Review the audit/figures, then fill review notes in cell 9 and run 9–10. A deliberate review gate prevents blind Run All from freezing the dataset. No model or training is included. Data and outputs remain private in Colab/Drive. After a reset, repeat from cell 1 using the SAME commit, dataset version and persistent directory.


## 1. Confirm Colab and choose the exact code revision
Use the full 40-character commit SHA of the reviewed Phase 2 code after its approved push. A CPU audit works too; preserve the already-working GPU PyTorch build.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re
from google.colab import drive  # Deliberately fails outside browser Colab.
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
REPOSITORY = 'https://github.com/hrb31/NeuroGuard.git'
REVISION = input('Full reviewed Git commit SHA: ').strip()
if not re.fullmatch(r'[0-9a-fA-F]{40}', REVISION):
    raise ValueError('Use a full commit SHA, not main or a short hash')
REVISION = REVISION.lower()
PRIVATE_REPOSITORY = False  # Change only if the repository requires authentication.
REPO = Path('/content/NeuroGuard')


## 2. Clone the source and verify the revision
Private repositories: set PRIVATE_REPOSITORY=True above, then grant the notebook access to a Colab Secret named GITHUB_TOKEN with read-only repository access, or enter it in the hidden prompt. Tokens are passed via a temporary Git askpass helper, never in URLs or notebook source.


In [ ]:
import tempfile
from getpass import getpass
env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'
with tempfile.TemporaryDirectory() as auth_dir:
    if PRIVATE_REPOSITORY:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = getpass('Read-only GitHub token (hidden): ')
        if not token:
            raise ValueError('Authentication required')
        helper = Path(auth_dir) / 'askpass.py'
        helper.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["NG_GITHUB_TOKEN"])\n')
        helper.chmod(0o700)
        env.update(GIT_ASKPASS=str(helper), NG_GITHUB_TOKEN=token)
        del token
    try:
        if not REPO.exists():
            subprocess.run(['git', 'clone', '--quiet', '--no-checkout', REPOSITORY, str(REPO)], env=env, check=True)
            subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', REVISION], env=env, check=True)
        actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
        if actual != REVISION:
            raise RuntimeError('Existing checkout differs. Save outputs, delete/reset the Colab runtime, and rerun. No local edits were overwritten.')
        if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
            raise RuntimeError('Checkout has local changes. Use a clean runtime.')
    finally:
        env.pop('NG_GITHUB_TOKEN', None)
        env.pop('GIT_ASKPASS', None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Executing source commit:', actual)


## 3. Install audit dependencies and run synthetic tests
These tests use generated arrays, not real patient data. They do not verify dataset counts or GPU training. Do not install the full application requirements over the working Colab torch build.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-audit.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## 4. Persistent private outputs
Grant Drive access in the browser. Keep this folder private: it will contain patient identifiers and dataset-derived examples. Raw MRI data stays in the temporary Colab cache and can be downloaded again after reset.


In [ ]:
drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/NeuroGuard/dataset-v2-seed42')
OUTPUT = PERSIST / 'runs' / REVISION
OUTPUT.mkdir(parents=True, exist_ok=True)
from src.data import write_json
versions = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(OUTPUT / 'environment.txt').write_text(versions)
write_json(OUTPUT / 'execution.json', {'commit': REVISION, 'python': sys.version,
           'torch': torch.__version__, 'cuda_available': torch.cuda.is_available()})
print('Private persistent output:', OUTPUT)


## 5. Verify release metadata and acknowledge terms
The setup metadata check found version 2, CC BY-NC-SA 4.0, sourced from TCIA/TCGA. Refresh metadata here; it is not a pixel audit. Read the [Kaggle release](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation), [licence](https://creativecommons.org/licenses/by-nc-sa/4.0/) and linked upstream terms. Downloads are pinned to version 2 even if a newer version appears. Do not redistribute images/weights under an assumed code licence.


In [ ]:
from src.acquisition import get_metadata, download_dataset
release = get_metadata()
write_json(OUTPUT / 'dataset_release.json', release)
info = release['metadata']
print(json.dumps({k: info.get(k) for k in ['title', 'currentVersionNumber', 'licenseName', 'lastUpdated', 'description']}, indent=2))
if info['currentVersionNumber'] != 2:
    raise RuntimeError('New release detected: review version-2 terms and record a decision before proceeding')
if info['licenseName'] != 'CC BY-NC-SA 4.0':
    raise RuntimeError('Licence metadata changed; review before download')
TERMS_ACCEPTED = input('After reading the applicable terms, type ACCEPT to download version 2: ') == 'ACCEPT'


## 6. Download into Colab only
KaggleHub is the supported downloader. Most public releases work without authentication. If consent/login is requested, add a Colab Secret KAGGLE_API_TOKEN from https://www.kaggle.com/settings/api and grant notebook access, or call kagglehub.login() for a hidden prompt, then rerun this cell. Never paste a token into code or send it back with outputs.


In [ ]:
DATA_ROOT = download_dataset(terms_accepted=TERMS_ACCEPTED)
print('Cloud dataset directory:', DATA_ROOT)


## 7. Audit every file and propose patient-disjoint assignments
No expected count is hardcoded as an observed result. The report includes corrupted/unpaired files, shapes/dtypes, intensity ranges, binary labels, channel equality, decoded-pixel duplicates and dataset hashes. Hashes detect exact duplicates, not all near-duplicates or incorrect annotations. Cross-patient nonblank duplicates must be resolved before freezing.


In [ ]:
from src.data import audit_dataset, split_patients, summarize, validate_split
audit = audit_dataset(DATA_ROOT)
write_json(OUTPUT / 'audit.json', audit)
print(json.dumps(audit['summary'], indent=2))
print('Integrity errors:', audit['errors'])
print('Duplicate groups:', len(audit['duplicate_images']))
print('Cross-patient duplicate groups:', len(audit['cross_patient_duplicates']))
print('Full private record:', OUTPUT / 'audit.json')
if audit['errors']:
    raise RuntimeError('Inspect audit.json; resolve all integrity errors before proceeding')
proposal = split_patients(audit['records'], seed=42)
split_summary = {s: summarize([r for r in proposal if r['split'] == s]) for s in ['train', 'validation', 'test']}
print(json.dumps(split_summary, indent=2))
write_json(OUTPUT / 'proposed_split_summary.json', split_summary)
print('Documentation/metadata files:', audit['summary']['other_files'])


## 8. Inspect training-only raw and processed examples
Show each stored channel separately, masks and overlays for tiny/large positive and empty examples; then deterministic versus augmented preprocessing. Channel semantics cannot be proved from colour appearance: inspect any release README/data dictionary and confirm the mapping from a primary source. Review all three channels; do not assume RGB. No test images are previewed.


In [ ]:
from src.visualization import save_previews
from IPython.display import display, Image
figure_paths = save_previews(DATA_ROOT, proposal, OUTPUT / 'figures')
for figure_path in figure_paths:
    display(Image(filename=figure_path))
# Read release text locally if present (do not print patient metadata tables).
for relative in audit['summary']['other_files']:
    if Path(relative).name.lower().startswith('readme') and Path(relative).suffix.lower() in {'.md', '.txt'}:
        print((DATA_ROOT / relative).read_text(errors='replace')[:12000])


## 9. Review and freeze the split
Fill CHANNEL_REVIEW with the source/mapping, observed channel equality, alignment and preprocessing assessment. If semantics remain unknown, say so and STOP before model development; retaining all stored channels is only a provisional transform. Record blank-image duplicate review if needed. Nonblank cross-patient copies need a documented cleaning decision and a fresh audit; do not bypass the gate. Never change the seed based on model performance. The frozen manifest is write-once and repeated runs must match it.


In [ ]:
CHANNEL_REVIEW = ''  # Fill after reviewing actual outputs and release documentation.
DUPLICATE_REVIEW = ''  # Explain any cross-patient all-zero background copies.
from src.data import freeze_split
manifest = freeze_split(DATA_ROOT, audit, PERSIST / 'split_manifest.json',
                        channel_review=CHANNEL_REVIEW, duplicate_review=DUPLICATE_REVIEW)
print(json.dumps(manifest['summary'], indent=2))
print('Training-only area tertiles:', manifest['size_cutoffs'])
print('Frozen manifest SHA256:', manifest['manifest_sha256'])
print('No model development until audit/figures/semantics and protocol have been reviewed.')


## 10. Save reproducibility evidence and return outputs
Send the printed audit/split summaries, test output, channel-review notes and preview figures back for verification. The private report bundle contains patient IDs: do not commit it or publish a saved Colab notebook with these outputs. Final submission outputs require privacy/licence review. Phase 2 is not complete merely because these cells were prepared.


In [ ]:
import hashlib, zipfile
write_json(OUTPUT / 'preprocessing.json', {'size': 256, 'channels': 'all stored channels pending semantic review',
    'normalisation': 'per-image/channel nonzero z-score; constant -> zero',
    'geometry': 'centred square padding; bilinear image, nearest mask',
    'augmentation': 'train only; horizontal flip p=.5; rotation +/-10 degrees p=.5',
    'seed': 42, 'channel_review': CHANNEL_REVIEW, 'manifest_sha256': manifest['manifest_sha256']})
bundle = PERSIST / ('phase2-evidence-' + REVISION[:12] + '.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(OUTPUT.rglob('*')):
        if item.is_file():
            archive.write(item, item.relative_to(OUTPUT))
    archive.write(PERSIST / 'split_manifest.json', 'split_manifest.json')
checksum = hashlib.sha256(bundle.read_bytes()).hexdigest()
bundle.with_suffix('.sha256').write_text(checksum + '  ' + bundle.name + '\n')
print('Saved private evidence bundle:', bundle)
print('SHA256:', checksum)
print('Stop here. No U-Net or training has been implemented.')
